In [ ]:
from  bs4 import BeautifulSoup
import requests
from urllib.parse import urlparse, urljoin
HEADERS = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            "Accept-Language": "es-ES,es;q=0.9",
        }

def extract_links(url, label=False, class_=False, id=False, formats=[]):
    # Extract all links from a desired element only and to the same domain
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
    except:
        return [], "error"
    soup = BeautifulSoup(response.text, 'html.parser')
    
    if class_:
        box = soup.find(label, class_=class_)
    elif id:
        box = soup.find(id=id)

    base_domain = urlparse(url).netloc  # current domain
    links = []
    if box:
        for a in box.find_all('a', href=True):
            href = a['href']
            absolute_url = urljoin(url, href)
            if urlparse(absolute_url).netloc == base_domain:
                links.append((absolute_url))
    if formats:
            links =  [link for link in links if link.split(".")[-1] in formats]
    # print(f"Extracted {len(links)} links from {url}")
    return links, soup

In [ ]:
url = "https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Orriak/default.aspx?page=1"
parsed = urlparse(url)
base_domain = f"{parsed.scheme}://{parsed.netloc}"

links = []
for n in range(2,10):
    print(f"visiting page: {url}")
    new_links, soup = extract_links(url = url, label="div", class_="AspNet-WebPartZone-Vertical", formats=["pdf"])
    links.extend(new_links)
    next_url = False
    for a in soup.find_all("a", class_="notCurrentPage"):
        if a.get_text(strip=True) == f"{n}":
            next_url = a
            break
    if next_url:
        url =  urljoin(base_domain, next_url["href"])
    else:
        break
for x in list(set(links)):
    print(x)

visiting page: https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Orriak/default.aspx?page=1
visiting page: https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Orriak/default.aspx?page=2
visiting page: https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Orriak/default.aspx?page=3
visiting page: https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Orriak/default.aspx?page=4
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/02_Revista Municipal Nº 2.pdf
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/17_Revista Municipal Nº 17.pdf
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/12_Revista Municipal Nº 12.pdf
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/09_Revista Municipal Nº 9.pdf
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/11_Revista Municipal Nº 11.pdf
https://www.ortuella.eus/eu-ES/Baliabideak/Argitalpenak/Udal Aldizkaria/13_Revista Municipal Nº